In [1]:
import sys
from pathlib import Path

from chromadb.api.client import SharedSystemClient

# Reset process-local clients after the persistent index was removed.
SharedSystemClient.clear_system_cache()

project_root = Path.cwd()
if not (project_root / "src").exists():
    for parent in project_root.resolve().parents:
        if (parent / "src").exists():
            project_root = parent
            break

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.core.config import Config
from src.document.chunker import chunk_sections
from src.document.pdf_parser import parse_pdf_document
from src.document.word_parser import parse_word_document
from src.retrieval.bm25_retriever import BM25Retriever
from src.retrieval.embedder import Embedder
from src.retrieval.hybrid_retriever import HybridRetriever
from src.retrieval.reranker import Reranker
from src.retrieval.vector_store import VectorStore

config = Config()

In [2]:
# Parse documents
sections = parse_pdf_document(project_root / "data/raw/HuMengqing.pdf", config)
sections += parse_word_document(project_root / "data/raw/Report.docx")
print(f"Loaded {len(sections)} section(s)")

# Split sections into chunks
chunks = chunk_sections(sections, config)
print(f"Created {len(chunks)} chunk(s)")

# Encode embeddings
embedder = Embedder(config)
chunk_embeddings = embedder.embed_chunks(chunks)
for chunk, embedding in zip(chunks, chunk_embeddings):
    chunk["embedding"] = embedding

# Store chunks in ChromaDB
vector_store = VectorStore(config, embedder=embedder)
vector_store.add_chunks(chunks)

# Build BM25 index
bm25 = BM25Retriever(chunks, config)

# Run retrieval
query = "What is the classification accuracy of ResNet26-V2?"
vector_results = vector_store.search(query, top_k=20)
bm25_results = bm25.search(query, top_k=20)

print("Vector results:")
for result in vector_results:
    print(result)

print("BM25 results:")
for result in bm25_results:
    print(result)


Loaded 123 section(s)
Created 187 chunk(s)


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Vector results:
{'chunk_id': 'HuMengqing_chunk_072', 'text': 'study has about 3.5 million parameters, whereas EfficientNet-B0, which is used for binary\nclassification tasks, has about 5.3 million parameters and VGG16 has 138 million\nparameters. It is worth noting that ResNet26-V2 achieves better performance with far\nfewer parameters than the other models, indicating that it significantly improves the\ncomputational efficiency and memory footprint while maintaining the accuracy of the\nmodel. This makes ResNet26-V2 a better choice for binary classification tasks in practical\napplications, especially in resource-limited environments.', 'metadata': {'source': 'HuMengqing.pdf', 'chunk_index': 72, 'page': 52, 'section_title': '4.4 Comparison with Other Convolutional Neural Networks', 'chunk_type': 'text'}, 'distance': 0.38927602767944336}
{'chunk_id': 'HuMengqing_chunk_099', 'text': '| Model | Accuracy | Recall<br>(Bad) | Precision<br>(Good) | F1 Score |\n| --- | --- | --- | --- | --- |

In [3]:
hybrid_retriever = HybridRetriever(vector_store, bm25, config)
hybrid_results = hybrid_retriever.search(query)

print(f"Hybrid results: {len(hybrid_results)}")
for rank, result in enumerate(hybrid_results, start=1):
    print(f"\n{rank}. {result['chunk_id']} | RRF: {result['rrf_score']:.6f}")
    print(
        f"Dense rank: {result['dense_rank']} | "
        f"BM25 rank: {result['bm25_rank']}"
    )
    print(result['metadata'])
    print(result['text'][:300])


Hybrid results: 10

1. HuMengqing_chunk_072 | RRF: 0.032787
Dense rank: 1 | BM25 rank: 1
{'source': 'HuMengqing.pdf', 'chunk_type': 'text', 'page': 52, 'chunk_index': 72, 'section_title': '4.4 Comparison with Other Convolutional Neural Networks'}
study has about 3.5 million parameters, whereas EfficientNet-B0, which is used for binary
classification tasks, has about 5.3 million parameters and VGG16 has 138 million
parameters. It is worth noting that ResNet26-V2 achieves better performance with far
fewer parameters than the other models, indi

2. HuMengqing_chunk_075 | RRF: 0.032002
Dense rank: 3 | BM25 rank: 2
{'chunk_type': 'text', 'section_title': '5 Summary', 'source': 'HuMengqing.pdf', 'page': 54, 'chunk_index': 75}
the infrastructure in this study, and the BottleneckV2 module is adopted for the design of
the network. To optimize the model performance, the width multiplication factor (K) and
the number of bottleneck modules (N) of the network are adjusted, respectively. After
sever

In [4]:
reranker = Reranker(config)
reranked_results = reranker.rerank(query, hybrid_results)

print(f"Reranked results: {len(reranked_results)}")
for rank, result in enumerate(reranked_results, start=1):
    print(
        f"\n{rank}. {result['chunk_id']} | "
        f"Rerank: {result['rerank_score']:.6f}"
    )
    print(f"RRF: {result['rrf_score']:.6f}")
    print(result['metadata'])
    print(result['text'][:300])

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Reranked results: 5

1. HuMengqing_chunk_072 | Rerank: 7.255227
RRF: 0.032787
{'source': 'HuMengqing.pdf', 'chunk_type': 'text', 'page': 52, 'chunk_index': 72, 'section_title': '4.4 Comparison with Other Convolutional Neural Networks'}
study has about 3.5 million parameters, whereas EfficientNet-B0, which is used for binary
classification tasks, has about 5.3 million parameters and VGG16 has 138 million
parameters. It is worth noting that ResNet26-V2 achieves better performance with far
fewer parameters than the other models, indi

2. HuMengqing_chunk_075 | Rerank: 5.132246
RRF: 0.032002
{'chunk_type': 'text', 'section_title': '5 Summary', 'source': 'HuMengqing.pdf', 'page': 54, 'chunk_index': 75}
the infrastructure in this study, and the BottleneckV2 module is adopted for the design of
the network. To optimize the model performance, the width multiplication factor (K) and
the number of bottleneck modules (N) of the network are adjusted, respectively. After
several rounds of experiment